In [12]:
import pyreadr
import torch

from src.gaussian_mixture import GaussianMixtureMAR
from src.metrics import energy_distance_faster

import numpy as np
import matplotlib.pyplot as plt

Load dataset.

In [13]:
datasets = ["parkinsons", "allergens", "concrete", "windspeed", "forest", "housing", "stock", "pumadyn32nm", "scm20d", "scm1d"]

In [14]:
def run_realdata(
        i, # dataset index
        B=50, # number of MC
        k_range=(1,500), # ks to choose from
        crit='bic', # criterion for choosing k
        ct='diag', # covariance structure
        n_inits=20 # number of inits
        ):
    print(datasets[i])
    Xstar_df = pyreadr.read_r(f"datasets/split_test/test.{datasets[i]}.RDS")[None]
    X_miss_df = pyreadr.read_r(f"datasets/split_amputed/mar.{datasets[i]}.RDS")[None]
    # Prepare the data
    Xm = torch.tensor(X_miss_df.values, dtype=torch.float64)
    X = torch.nan_to_num(Xm, nan=0.0) # input to GMM
    Xstar = torch.tensor(Xstar_df.values)
    M_np = (X_miss_df.notna()).astype(int).values
    M = torch.tensor(M_np, dtype=torch.float64)
    n_samples = X.shape[0]
    gmm = GaussianMixtureMAR(
                k_range=k_range, 
                criterion=crit,
                cov_type=ct, 
                n_init=n_inits
            )
    gmm.fit(X, M)
    ed_scaled = [energy_distance_faster(Xstar, gmm.sample(n_samples)[0], scale=True) for _ in range(B)]
    return ed_scaled

In [15]:
res = np.array(
    [run_realdata(i) for i in range(len(datasets))]
)

parkinsons
  k=192 -> score=-26117.7891
  k=309 -> score=105.5547
  k=119 -> score=-42591.5508
  k=74 -> score=-52408.2695
  k=46 -> score=-58121.0742
  k=29 -> score=-60910.3633
  k=18 -> score=-61965.5977
  k=12 -> score=-61208.5195
  k=23 -> score=-61574.4961
  k=16 -> score=-62000.4062
  k=14 -> score=-61858.0391
  k=17 -> score=-61951.9922
allergens
  k=192 -> score=2760.5625
  k=309 -> score=49490.2344
  k=119 -> score=-12133.9844
  k=74 -> score=-15689.5273
  k=46 -> score=-11378.0645
  k=91 -> score=-20857.3242
  k=102 -> score=-11633.1641
  k=85 -> score=-14989.6758
  k=96 -> score=-18070.0078
  k=89 -> score=-16754.3867
  k=93 -> score=-18459.2969
  k=92 -> score=-15860.6055
concrete
  k=192 -> score=22595.9336
  k=309 -> score=30114.2578
  k=119 -> score=18158.6133
  k=74 -> score=18089.4805
  k=46 -> score=18578.8809
  k=91 -> score=18273.8594
  k=63 -> score=18032.9395
  k=57 -> score=18152.0762
  k=68 -> score=18403.2324
  k=61 -> score=18337.7812
  k=65 -> score=18713.68

In [16]:
np.save('realdata_results.npy', res)

In [61]:
import pandas as pd

In [63]:
means = res.mean(axis=1)
vars = res.std(axis=1)
res_df = pd.DataFrame({datasets[i]:[round(means[i].tolist(),2), round(vars[i].tolist(),2)] for i in range(10)}, index=['mean', 'std'])

In [65]:
cols = ['scm1d', 'scm20d', 'pumadyn32nm', 'parkinsons', 'allergens', 'concrete', 'stock', 'forest', 'housing', 'windspeed']
res_df[cols].T

,mean,std
scm1d,75.04,5.14
scm20d,46.76,4.54
pumadyn32nm,11.92,1.26
parkinsons,14245.72,328.30
allergens,40.67,8.25
concrete,7.60,1.47
stock,7.78,2.15
forest,5.51,1.30
housing,9.66,3.19
windspeed,3.43,0.82
